# Ứng dụng 2: tìm motif bảo tồn trong họ protein kinase

Protein kinase là một trong những họ protein lớn và quan trọng nhất ở người: chúng gắn nhóm phosphate
lên protein khác để bật/tắt tín hiệu trong tế bào, và là đích của nhiều thuốc điều trị ung thư.
Dù rất đa dạng, các kinase có chung một miền xúc tác với vài đoạn amino acid gần như không đổi qua
tiến hóa (motif bảo tồn).

Notebook này hỏi: **chỉ dùng khai thác tập phổ biến, không gióng hàng chuỗi (alignment), không dùng
kiến thức sinh học cho trước, có tìm lại được các motif đó không?** Nếu có, kết quả kiểm chứng được
bằng tài liệu sinh học. Mọi tập phổ biến và luật đều do cài đặt Apriori của nhóm (`AprioriFIM`) tính.

**Chạy trên Colab:** *Runtime → Change runtime type → Julia*, rồi *Runtime → Run all* (khoảng 1–2 phút).

## 1. Cài đặt

In [2]:
# Tìm thư mục repo: mở notebook ngay trong repo (thư mục gốc hoặc notebooks/) thì dùng luôn;
# trên Colab thì shallow clone về. Gói chỉ dùng thư viện chuẩn nên không cần tải registry của Pkg.
is_repo(dir) = isfile(joinpath(dir, "src", "AprioriFIM.jl"))
const REPO = abspath(is_repo(".") ? "." : is_repo("..") ? ".." : "FIM_Group14")
is_repo(REPO) || run(`git clone --depth 1 https://github.com/TaVanDuc2005/FIM_Group14`)
using Pkg
Pkg.activate(REPO)
using AprioriFIM
include(joinpath(REPO, "applications", "bioinformatics.jl"))   # đọc FASTA, tách k-mer, in bảng
pct(x) = string(round(100x; digits = 1), "%")

  Activating project at `/content/FIM_Group14`


pct (generic function with 1 method)

## 2. Dữ liệu

Hai tập protein người lấy từ UniProt Swiss-Prot (đã được chuyên gia kiểm duyệt), bản 2026_04:

- **kinase**: mọi protein người có miền Pfam PF00069 (miền kinase serine/threonine kinh điển).
- **nền**: 1 000 protein người chọn ngẫu nhiên (seed 14) trong số các protein *không* phải kinase.
  Mọi protein có từ khóa Kinase hoặc có miền kinase (kể cả tyrosine kinase, PF07714) đều bị loại,
  vì chúng cũng mang các motif đang tìm và sẽ làm sai phép so sánh.

Tập nền đóng vai trò đối chứng: một đoạn amino acid hay gặp ở kinase chỉ thú vị khi nó *không* hay gặp
ở protein bất kỳ. Script tải dữ liệu là `applications/prepare_protein_data.jl`; nguồn và câu truy vấn
ghi trong `data/bio/README.md`. Dữ liệu được lưu sẵn trong repo để kết quả không đổi khi UniProt cập nhật.

In [3]:
kinases = read_fasta(joinpath(REPO, "data", "bio", "kinase.fasta"))
background = read_fasta(joinpath(REPO, "data", "bio", "background.fasta"))
median_length(rs) = sort([length(r.sequence) for r in rs])[cld(length(rs), 2)]
print_table(["Nhóm", "Số protein", "Độ dài trung vị (amino acid)"],
            [("kinase", length(kinases), median_length(kinases)),
             ("nền", length(background), median_length(background))])
println()
println("Ví dụ: ", kinases[1].entry, " - ", first(kinases[1].description, 60), "...")
println(first(kinases[1].sequence, 60), "...")

Nhóm    Số protein  Độ dài trung vị (amino acid)
------  ----------  ----------------------------
kinase  344         588                         
nền     1000        394                         

Ví dụ: SGK1_HUMAN - Serine/threonine-protein kinase Sgk1 OS=Homo sapiens OX=9606...
MTVKTEAAKGTLTYSRMRGMVAILIAFMKQRRMGLNDFIQKIANNSYACKHPEVQSILKI...


## 3. Chuyển protein thành giao dịch

Một chuỗi protein là một dãy chữ cái (20 loại amino acid), không có sẵn "item". Nhóm thiết kế như sau:

- **Mỗi protein là một giao dịch.** Support của một mẫu khi đó là số protein chứa mẫu đó, đúng thứ
  ta muốn đo khi nói một motif "bảo tồn" trong họ.
- **Item là các k-mer**, tức các đoạn liền dài k amino acid có trong chuỗi. Ví dụ `MKTAY` với k = 3
  cho các item `MKT`, `KTA`, `TAY`.
- **Giao dịch là tập hợp**: một k-mer xuất hiện nhiều lần trong chuỗi vẫn chỉ tính một lần, và vị trí
  bị bỏ qua. Cái giá là mất thông tin vị trí; mục 5 kiểm tra lại vị trí cho các motif tìm được.

**Chọn k.** k quá nhỏ thì đoạn nào cũng gặp ở mọi protein; k quá lớn thì gần như không đoạn nào lặp lại
giữa các protein. Bảng dưới đo trên tập kinase, minsup 50% (chỉ khai thác đến 2-itemset cho nhanh):

In [4]:
rows = []
for k in (2, 3, 4)
    tx_k, labels_k = encode_transactions(protein_transactions(kinases; k))
    itemsets_k = apriori_bitarray(tx_k, 0.5; maxlen = 2)
    push!(rows, (k, length(labels_k), round(Int, sum(length, tx_k) / length(tx_k)),
                 count(s -> length(s) == 1, keys(itemsets_k)), count(s -> length(s) == 2, keys(itemsets_k))))
end
print_table(["k", "số k-mer khác nhau", "k-mer TB mỗi protein", "1-itemset phổ biến", "2-itemset phổ biến"], rows)

k  số k-mer khác nhau  k-mer TB mỗi protein  1-itemset phổ biến  2-itemset phổ biến
-  ------------------  --------------------  ------------------  ------------------
2  400                 269                   304                 35559             
3  7895                665                   9                   5                 
4  91935               798                   0                   0                 


- **k = 2**: chỉ có 400 cặp amino acid, một protein dài vài trăm amino acid chứa gần hết. Hàng trăm
  item và hàng chục nghìn cặp "phổ biến" chỉ phản ánh thành phần amino acid, không phải motif.
  Dữ liệu dày đến mức khai thác đầy đủ không chạy xong (lý do có tham số `maxlen`).
- **k = 4**: có gần 92 nghìn 4-mer khác nhau; một motif ngắn bị đột biến ở bất kỳ vị trí nào cũng tách
  thành nhiều 4-mer khác nhau, nên không 4-mer nào đạt 50%.
- **k = 3** cho vài item phổ biến, đủ ít để đọc hết và đủ dài để mang tính đặc hiệu. Nhóm chọn **k = 3**.

## 4. Khai thác tập phổ biến trong họ kinase

Bảng dưới cho thấy số tập phổ biến theo minsup. Dữ liệu sinh học dày (mỗi protein có khoảng 665
item), nên khi giảm minsup, số tập tăng nhanh. Phần phân tích dùng **minsup = 40%**: một motif phải có
mặt ở ít nhất 138 trong 344 kinase.

In [5]:
tx, labels = encode_transactions(protein_transactions(kinases; k = 3))
rows = []
for minsup in (0.5, 0.4, 0.3, 0.25)
    t = @elapsed found = apriori_bitarray(tx, minsup)
    push!(rows, (pct(minsup), length(found), maximum(length, keys(found)), round(1000t; digits = 1)))
end
print_table(["minsup", "số tập phổ biến", "tập dài nhất", "thời gian (ms)"], rows)

itemsets = apriori_bitarray(tx, 0.4)

minsup  số tập phổ biến  tập dài nhất  thời gian (ms)
------  ---------------  ------------  --------------
50.0%   14               2             20.6          
40.0%   47               3             21.3          
30.0%   240              4             36.3          
25.0%   723              5             136.7         


Dict{Vector{Int64}, Int64} with 47 entries:
  [975]            => 218
  [2658, 5600]     => 149
  [872, 6133]      => 139
  [975, 2658]      => 144
  [872, 1375]      => 147
  [872, 975]       => 184
  [872, 975, 5600] => 144
  [1694]           => 146
  [2158]           => 151
  [4829]           => 178
  [244, 5600]      => 156
  [2310, 2658]     => 138
  [3749]           => 155
  [2310]           => 189
  [3739]           => 174
  [872, 4829]      => 153
  [975, 5600]      => 172
  [3755]           => 168
  [2962]           => 147
  [4504]           => 144
  [872, 2310]      => 171
  [872, 3739]      => 148
  [244]            => 247
  [244, 872]       => 207
  [2658]           => 238
  ⋮                => ⋮

**Các 3-mer phổ biến.** So tỉ lệ có mặt trong kinase với trong nền. Cột cuối là số lần
"hay gặp hơn" ở kinase so với nền:

In [6]:
singles = sort([(only(decode(s, labels)), c / length(kinases)) for (s, c) in itemsets if length(s) == 1];
               by = p -> -p[2])
print_table(["3-mer", "trong kinase", "trong nền", "kinase / nền"],
            [(m, pct(f), pct(motif_fraction(background, m)),
              round(f / max(motif_fraction(background, m), 1 / length(background)); digits = 1))
             for (m, f) in singles])

3-mer  trong kinase  trong nền  kinase / nền
-----  ------------  ---------  ------------
DFG    80.5%         5.5%       14.6        
APE    71.8%         10.8%      6.6         
HRD    69.2%         3.3%       21.0        
DLK    63.4%         12.4%      5.1         
RDL    58.1%         13.4%      4.3         
GTP    54.9%         10.4%      5.3         
ELL    53.2%         26.4%      2.0         
PEV    51.7%         11.7%      4.4         
LKP    50.6%         13.2%      3.8         
LLK    48.8%         21.9%      2.2         
SLG    48.3%         23.2%      2.1         
LLL    46.2%         38.7%      1.2         
SSS    45.6%         31.2%      1.5         
LLD    45.1%         21.9%      2.1         
LLG    44.8%         29.9%      1.5         
GLA    43.9%         18.8%      2.3         
ILL    42.7%         19.1%      2.2         
FGL    42.4%         9.0%       4.7         
LLS    42.2%         34.0%      1.2         
NIL    41.9%         9.0%       4.7         
LSS    41.

**Các tập nhiều 3-mer.** Một số tập gồm các 3-mer chồng lên nhau (ví dụ `HRD` và `RDL` chồng 2 chữ),
nối lại được thành một đoạn dài hơn. Cột thứ ba ghi đoạn đó, cột cuối kiểm tra đoạn liền đó có hay gặp
ở protein nền không.

In [7]:
multi = sort([(decode(s, labels), c) for (s, c) in itemsets if length(s) >= 2]; by = p -> (-p[2], p[1]))
rows = []
for (s, c) in multi
    chain = chain_kmers(s)
    push!(rows, (join(s, ", "), pct(c / length(kinases)),
                 chain === nothing ? "" : chain,
                 chain === nothing ? "" : string(pct(motif_fraction(kinases, chain)), " / ", pct(motif_fraction(background, chain)))))
end
print_table(["itemset", "support (kinase)", "nối thành đoạn liền", "đoạn liền: kinase / nền"], rows)

itemset        support (kinase)  nối thành đoạn liền  đoạn liền: kinase / nền
-------------  ----------------  -------------------  -----------------------
DFG, HRD       61.3%                                                         
APE, DFG       60.2%                                                         
DFG, DLK       53.5%                                                         
APE, HRD       50.0%                                                         
DLK, RDL       50.0%             RDLK                 44.8% / 0.4%           
DFG, GTP       49.7%                                                         
APE, DLK       49.4%                                                         
DFG, RDL       47.7%                                                         
APE, RDL       45.3%                                                         
DFG, PEV       44.5%                                                         
APE, DFG, HRD  44.2%                                            

### Diễn giải

Thuật toán không biết gì về sinh học, nhưng các 3-mer nó tìm ra khớp với những motif mà Hanks và Hunter
(1995) mô tả khi so sánh hàng trăm kinase. Miền kinase được chia thành 12 tiểu vùng (subdomain), và các
motif bảo tồn nhất nằm ở:

| 3-mer tìm được | Motif đã biết | Vai trò |
|---|---|---|
| `HRD`, `RDL`, `DLK`, `LKP` | **HRDLKPEN**, vòng xúc tác (subdomain VIb) | Asp (D) của HRD nằm sát nhóm OH của amino acid được gắn phosphate, tham gia trực tiếp vào phản ứng xúc tác |
| `DFG` | **DFG** (subdomain VII) | giúp giữ ATP và ion Mg²⁺ ở trung tâm hoạt động; mở đầu đoạn kích hoạt (activation segment) |
| `APE`, `GTP`, `PEV` | **…GTPxYxAPE…** (subdomain VIII) | APE đánh dấu cuối đoạn kích hoạt |

Những điểm đáng chú ý:

- `DFG` có mặt ở khoảng 80% kinase nhưng chỉ khoảng 5% protein nền, và `HRD` còn đặc hiệu hơn (69% so
  với 3%). Chúng không đạt 100% vì một số kinase mang biến thể khác ở các vị trí này: chỉ cần đổi 1 amino
  acid là 3-mer đã thành một item khác.
- Các tập như {DLK, RDL} nối thành `RDLK`, có ở gần một nửa số kinase nhưng gần như không có ở nền
  (dưới 1%). Thuật toán đã "ghép" lại mảnh của motif HRDLKPEN từ các item ngắn.
- Tập {DFG, HRD} có support khoảng 60%: hai motif ở hai tiểu vùng khác nhau thường đi cùng nhau, vì cả
  hai đều cần cho chức năng xúc tác. Mục 5 cho thấy chúng còn nằm ở khoảng cách cố định.
- Một số 3-mer phổ biến như `LLL`, `SSS` *không* đặc trưng cho kinase: chúng gặp ở cả nền với tỉ lệ gần
  bằng. Đây là các vùng lặp đơn giản (low-complexity) có ở mọi loại protein. Support cao chưa đủ để nói
  một mẫu có ý nghĩa; mục 6 dùng lift để tách hai loại này.

## 5. Kiểm tra vị trí: HRD và DFG có nằm trong cùng một vùng không?

Giao dịch đã bỏ qua vị trí, nên itemset {DFG, HRD} chỉ nói hai đoạn cùng có mặt. Quay lại chuỗi gốc,
đo khoảng cách từ HRD tới DFG gần nhất phía sau:

In [8]:
spacing = Int[]
for r in kinases
    h = findfirst("HRD", r.sequence)
    h === nothing && continue
    d = findnext("DFG", r.sequence, last(h))
    d === nothing || push!(spacing, first(d) - first(h))
end
sort!(spacing)
quantile_of(p) = spacing[clamp(round(Int, p * length(spacing)), 1, length(spacing))]
println(length(spacing), " kinase có HRD và DFG phía sau")
println("khoảng cách trung vị: ", quantile_of(0.5), " amino acid (25%-75%: ", quantile_of(0.25), "-", quantile_of(0.75), ")")
println("trong khoảng 18-23 amino acid: ", pct(count(d -> 18 <= d <= 23, spacing) / length(spacing)))

206 kinase có HRD và DFG phía sau
khoảng cách trung vị: 20 amino acid (25%-75%: 20-21)
trong khoảng 18-23 amino acid: 87.9%


Khoảng cách (tính từ chữ H của HRD tới chữ D của DFG) gần như cố định ở **20 amino acid**. Kinase PKA,
protein mẫu dùng để nghiên cứu cấu trúc kinase, cũng cho đúng con số này: vòng xúc tác của nó là biến thể
YRD ở vị trí 164–166, còn Asp của DFG ở vị trí 184. Hai motif là hai phần của cùng một cấu trúc không gian
quanh ATP, nên khoảng cách giữa chúng bị ràng buộc chặt qua tiến hóa. Kết quả này củng cố rằng itemset {DFG, HRD} phản ánh một đơn vị cấu trúc thật, không
phải hai đoạn tình cờ cùng có mặt.

## 6. Luật kết hợp: motif nào đặc trưng cho kinase?

Gộp kinase và nền thành một CSDL, thêm vào mỗi giao dịch một item nhãn (`KINASE` hoặc `OTHER`), rồi sinh
luật. Luật dạng `{3-mer...} ⇒ {KINASE}` trả lời: *thấy các đoạn này thì protein có phải kinase không?*

- **confidence**: trong các protein chứa vế trái, bao nhiêu phần trăm là kinase.
- **lift**: confidence chia cho tỉ lệ kinase trong CSDL (344/1344 ≈ 25,6%). Lift tối đa là 1344/344 ≈ 3,91,
  đạt khi mọi protein chứa vế trái đều là kinase. Lift gần 1 nghĩa là vế trái không nói gì về nhãn.

Dữ liệu trộn có minsup 10% (134 protein); khai thác giới hạn ở tập tối đa 4 item (`maxlen = 4`) để đủ
nhanh trên Colab.

In [9]:
mixed = vcat(protein_transactions(kinases; label = "KINASE"), protein_transactions(background; label = "OTHER"))
txm, labm = encode_transactions(mixed)
n = length(txm)
kinase_id = findfirst(==("KINASE"), labm)
@time itemsets_m = apriori_bitarray(txm, 0.1; maxlen = 4)
@time rules = association_rules(itemsets_m, n; minconf = 0.0)
strong = filter(r -> r.consequent == [kinase_id] && r.confidence >= 0.8, rules)
println(length(itemsets_m), " tập phổ biến; ", count(r -> r.confidence >= 0.8, rules),
        " luật có confidence ≥ 80%, trong đó ", length(strong), " luật ⇒ KINASE")
println("tỉ lệ kinase trong CSDL: ", pct(length(kinases) / n), ", lift tối đa: ", round(n / length(kinases); digits = 2))
println()
println("Top 10 luật ⇒ KINASE theo lift:")
foreach(r -> println("  ", format_rule(r, labm)), first(sort(strong; by = r -> (-r.lift, -r.support)), 10))

  2.140501 seconds (19.56 M allocations: 611.207 MiB, 30.61% gc time)
  0.210567 seconds (293.40 k allocations: 13.709 MiB, 95.76% compilation time)
4517 tập phổ biến; 60 luật có confidence ≥ 80%, trong đó 29 luật ⇒ KINASE
tỉ lệ kinase trong CSDL: 25.6%, lift tối đa: 3.91

Top 10 luật ⇒ KINASE theo lift:
  {APE, DFG, HRD} => {KINASE}  sup = 152, conf = 0.993, lift = 3.881
  {DFG, HRD} => {KINASE}  sup = 211, conf = 0.991, lift = 3.87
  {DFG, DLK, RDL} => {KINASE}  sup = 144, conf = 0.98, lift = 3.827
  {APE, DFG, DLK} => {KINASE}  sup = 143, conf = 0.979, lift = 3.827
  {APE, DFG, GTP} => {KINASE}  sup = 137, conf = 0.979, lift = 3.823
  {APE, DFG} => {KINASE}  sup = 207, conf = 0.958, lift = 3.744
  {APE, DLK, RDL} => {KINASE}  sup = 137, conf = 0.958, lift = 3.743
  {APE, HRD} => {KINASE}  sup = 172, conf = 0.956, lift = 3.733
  {DLK, HRD} => {KINASE}  sup = 144, conf = 0.954, lift = 3.726
  {DFG, PEV} => {KINASE}  sup = 153, conf = 0.95, lift = 3.713


**Một motif riêng lẻ so với tổ hợp motif**, và so với các 3-mer chỉ phổ biến mà không đặc trưng:

In [10]:
function rule_to_kinase(kmers)
    ids = sort([findfirst(==(x), labm) for x in kmers])
    i = findfirst(r -> r.antecedent == ids && r.consequent == [kinase_id], rules)
    return i === nothing ? nothing : rules[i]
end
rows = []
for kmers in (["DFG"], ["HRD"], ["APE"], ["DFG", "HRD"], ["APE", "DFG", "HRD"], ["LLL"], ["SSS"])
    r = rule_to_kinase(kmers)
    r === nothing && continue
    push!(rows, (join(kmers, ", "), r.support, pct(r.confidence), round(r.lift; digits = 2)))
end
print_table(["vế trái (⇒ KINASE)", "support", "confidence", "lift"], rows)

vế trái (⇒ KINASE)  support  confidence  lift
------------------  -------  ----------  ----
DFG                 277      83.4%       3.26
HRD                 238      87.8%       3.43
APE                 247      69.6%       2.72
DFG, HRD            211      99.1%       3.87
APE, DFG, HRD       152      99.3%       3.88
LLL                 159      29.1%       1.14
SSS                 157      33.5%       1.31


### Diễn giải

- **Tổ hợp mạnh hơn từng motif.** Chỉ thấy `DFG` thì khoảng 83% là kinase; thấy cả `DFG` và `HRD` thì
  tỉ lệ lên khoảng 99%, lift gần mức tối đa. Một 3-mer riêng lẻ vẫn có thể xuất hiện tình cờ trong
  protein bất kỳ, nhưng hai motif của cùng miền xúc tác cùng có mặt thì gần như chắc chắn là kinase.
  Đây chính là thông tin mà itemset mang lại so với việc chỉ đếm từng k-mer.
- **Support cao không đồng nghĩa với có ý nghĩa.** `LLL` và `SSS` có support tương đương các motif
  thật, nhưng lift chỉ quanh 1,1–1,3: chúng gặp ở kinase gần như ở mọi protein. Nếu chỉ xếp theo
  support, chúng sẽ lẫn vào kết quả; lift loại chúng ra.
- **Ứng dụng.** Các luật này hoạt động như một bộ nhận diện đơn giản, dễ giải thích: với một protein
  mới chưa rõ chức năng, nếu chứa {DFG, HRD} thì nhiều khả năng nó là kinase, và có thể chỉ ra đúng vị
  trí của vòng xúc tác. Cách làm không cần gióng hàng chuỗi, nên áp dụng được cho các họ protein khác
  chỉ bằng cách đổi dữ liệu.

## 7. Hạn chế

- **Motif có khoảng trống không bắt được.** Motif gắn ATP `GxGxxG` (subdomain I) có các vị trí tùy ý ở
  giữa, nên không thành một k-mer cố định; nó không xuất hiện trong kết quả dù rất bảo tồn.
- **Bỏ qua vị trí và số lần lặp.** Hai 3-mer cùng có mặt chưa chắc nằm gần nhau; mục 5 phải kiểm tra
  lại bằng chuỗi gốc.
- **k cố định.** Một đột biến ở giữa motif làm mất cả 3 k-mer phủ qua nó, nên support của motif bị
  đánh giá thấp (DFG chỉ khoảng 80% dù gần như mọi kinase hoạt động đều có motif này hoặc biến thể của nó).
- **Dữ liệu chỉ gồm protein người** và một mẫu nền 1 000 protein; tỉ lệ ở nền có sai số lấy mẫu.

**Tài liệu tham khảo.** S. K. Hanks and T. Hunter, "The eukaryotic protein kinase superfamily: kinase
(catalytic) domain structure and classification," *FASEB J.*, vol. 9, no. 8, pp. 576–596, 1995.